# Assessing an Observed Galaxy

This notebook prepares a BayMorph analysis of one observed galaxy. The Advisor
reads the FITS image and its header, you confirm the target and declare the
scientific assumptions, and the notebook writes one reviewed YAML
configuration.

It does not prepare the image or run the fit: `02B_Observed_Fit.ipynb` reads
the YAML and runs the whole pipeline. The YAML is the contract between the two
notebooks.

BayMorph never derives priors from the image it will fit. The image-derived
quantities shown here are diagnostics.

In [ ]:
from pathlib import Path

import yaml

from baymorph import ObservationAdvisor

## Observation

Relative paths are resolved from the project root, so the notebook also works
from the `notebooks` directory. Replace the FITS path with your own image.
`analysis_name` names both the YAML and the run directory.

In [ ]:
fits_path = Path("data/galaxy_r.fits")
analysis_name = "galaxy_r"
config_path = Path("configs") / f"{analysis_name}.yaml"

advisor = ObservationAdvisor.from_project(fits_path)

print(f"FITS: {advisor.fits_path}")
print(f"Project root: {advisor.project_root}")

## Target

Click the target galaxy, then press **Confirm center**. BayMorph uses
zero-based pixel coordinates: `x` is the NumPy column and `y` the NumPy row.
A candidate position is never accepted implicitly.

Without the interactive widgets, set the centre explicitly instead with
`advisor.set_target_position(x, y)`.

In [ ]:
advisor.select_target(
    title="Select the target centre",
)

## Header and image diagnostics

The summary reports the header metadata BayMorph can read and diagnostics for
the confirmed target.

The background and noise scales here are measured over the whole image. The
noise scale that enters the likelihood is measured later, on the prepared
cutout. The saturation keyword is shown as written in the header: its units
are not checked against the image.

In [ ]:
if advisor.target_position is None:
    raise ValueError(
        "Confirm the target centre in the selector before continuing."
    )

advisor.inspect()

print(advisor.summary())

## Prior guidance

For each single-Sérsic parameter the Advisor names a prior family and, where
it can, an observed scale. Observed scales come from the image being fitted:
use them to check that a declared prior is sensible, never to set its bounds.

In [ ]:
guidance = advisor.prior_guidance("single_sersic")

for name, item in guidance.items():
    print(f"{name}: {item.family}")

    if item.observed_scale is not None:
        print(f"  observed scale (diagnostic): {item.observed_scale:.6g}")

    print(f"  {item.rationale}")

## Scientific declarations

Declare each assumption from a policy you can justify without this image's
fitted intensities, for example a survey magnitude range converted to image
units. Values are in the image units reported above (`BUNIT`).

- `flux_prior`: the total-flux prior of the starting model.
- `model_priors`: overrides for the other morphology priors. Those left out
  are BayMorph reference defaults, named in the proposal warnings and written
  into the YAML.
- `background`: the residual background left after preparation removes the
  sky. `constant` samples `background_level`; `plane` also samples
  `background_gradient_x` and `background_gradient_y`, and needs a prior for
  each.
- `effective_gain`: where the effective gain of the source Poisson variance
  comes from (AD-047): `declared` with `gain`; `sky_level` with the level of
  a sky subtracted before BayMorph, for example from a header keyword;
  `measured_sky` when the sky is still in the image; or `absent`. Both sky
  sources require the sky level to be at least ten times the sky scale.
- `psf_max_peak`: the brightest peak pixel accepted for an empirical-PSF star.
  Keep it below the detector saturation level, in image units, so that
  saturated stars cannot enter the PSF.
- `cutout_mode`: `auto` sizes the cutout from the galaxy's extent and can be
  very large for an extended source; `fixed` uses a square of `fixed_size`
  pixels. The cutout is the likelihood support, so it is part of the
  scientific problem. Preparation rejects a cutout that does not contain the
  sky plateau around the galaxy plus a margin.

`single_sersic` is the simplest starting model, not a claim that the galaxy is
a single Sérsic profile. The cell stops until the three declarations are filled
in.

In [ ]:
starting_model = "single_sersic"

flux_prior = None
# flux_prior = {"type": "log_uniform", "lower": ..., "upper": ...}

model_priors = {}
# model_priors = {"re": {"type": "uniform", "lower": ..., "upper": ...}}

background = None
# background = {
#     "model": "constant",
#     "priors": {
#         "background_level": {
#             "type": "uniform",
#             "lower": ...,
#             "upper": ...,
#         },
#     },
# }

effective_gain = None
# effective_gain = {"source": "sky_level", "sky_level": ...}

psf_max_peak = None

cutout_mode = "auto"
fixed_size = None

missing = [
    name
    for name, value in (
        ("flux_prior", flux_prior),
        ("background", background),
        ("effective_gain", effective_gain),
        ("psf_max_peak", psf_max_peak),
    )
    if value is None
]

if missing:
    raise ValueError(
        "Declare these before proposing a configuration: " + ", ".join(missing)
    )

## Computational settings

These settings change how the posterior is computed, not the scientific
problem. Nautilus returns weighted posterior samples and the Bayesian
evidence, and BayMorph fills in its remaining defaults. `filepath: auto` keeps
a signed checkpoint in the run directory, so an interrupted fit resumes
instead of restarting. The likelihood is the noise model of AD-047, a correlated Gaussian
evaluated through a nearest-neighbour factorization, with the effective gain
declared above.

In [ ]:
sampler = {
    "name": "nautilus",
    "init": {
        "seed": 42,
        "filepath": "auto",
    },
}

likelihood = {
    "type": "correlated_gaussian",
    "include_normalization": True,
    "effective_gain": effective_gain,
}

## Review the proposal

The proposal is built in memory and nothing is written yet. Read the warnings
and the complete YAML before accepting it.

In [ ]:
proposal = advisor.propose_config(
    output_path=config_path,
    model_name=starting_model,
    flux_prior=flux_prior,
    model_priors=model_priors,
    background=background,
    psf_max_peak=psf_max_peak,
    cutout_mode=cutout_mode,
    fixed_size=fixed_size,
    object_name=analysis_name,
    likelihood=likelihood,
    sampler=sampler,
)

print(proposal.summary())
print()
print(yaml.safe_dump(proposal.config, sort_keys=False))

## Write the YAML

`accept=True` records that you reviewed the proposal. BayMorph refuses to
replace an existing YAML. To change a declaration after 02B has run, choose a
new `analysis_name`: products already written from the old YAML are never
updated.

In [ ]:
written_path = proposal.write(accept=True)

print(f"Configuration written to: {written_path}")
print("Next: open 02B_Observed_Fit.ipynb.")